# MichiganCast Demo 04: Imbalance Strategy Stability Export and Serving

This notebook demonstrates a production oriented closure loop from imbalance optimization to model serving observability.

## 0. Objectives and Conclusions

**Objectives**
1. Compare multiple class imbalance strategies.
2. Validate training stability with repeated runs.
3. Export a trained model for standalone inference.
4. Verify API inference and runtime monitoring outputs.

**Conclusion Template**
- Which imbalance strategy improved positive class metrics.
- Whether run to run metric variance is within acceptance bounds.
- Whether export and serving paths are operational.

## 1. Inputs and Outputs

| Type | Path |
|---|---|
| Imbalance experiment | `src/train/imbalance.py` |
| Stability training | `src/train/train.py` |
| Export module | `src/train/export.py` |
| Standalone inference | `src/serve/infer_torchscript.py` |
| API and monitoring | `src/serve/app.py`, `src/serve/monitoring.py` |
| Imbalance report | `artifacts/reports/imbalance_strategy_comparison.*` |
| Stability report | `artifacts/reports/stability_train_report*.json` |
| Exported model | `artifacts/models/*.ts` |
| Inference output | `artifacts/models/inference_output*.json` |
| Monitoring log | `artifacts/logs/inference_monitoring.jsonl` |

In [ ]:
from pathlib import Path
import subprocess
import json
import pandas as pd

paths_to_check = [
    'src/train/imbalance.py',
    'src/train/train.py',
    'src/train/export.py',
    'src/serve/infer_torchscript.py',
    'src/serve/app.py',
    'src/serve/monitoring.py',
]

for p in paths_to_check:
    print(f'{p}:', 'OK' if Path(p).exists() else 'MISSING')

## 2. Method and Implementation Using src Modules

Commands are printed by default to prevent blocking runs. Set `DEMO_EXECUTE=True` to execute.

In [ ]:
DEMO_EXECUTE = False

def run_or_print(cmd: str):
    print('\n$ ' + cmd)
    if not DEMO_EXECUTE:
        print('[skip] DEMO_EXECUTE=False, only showing command.')
        return None
    proc = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
    return proc.returncode

### 2.1 Class Imbalance Experiments T23

**Purpose** Compare baseline BCE, weighted BCE, focal loss, and threshold moving under the same split and metric protocol.

In [ ]:
cmd_imbalance = (
    'scripts/run_in_pytorch_env.sh python -m src.train.imbalance '
    '--input data/interim/traverse_city_daytime_clean_v1.csv '
    '--output-dir artifacts/reports '
    '--epochs 20 '
    '--batch-size 256 '
    '--device cpu'
)
run_or_print(cmd_imbalance)

### 2.2 Stability Dual Run T24

**Purpose** Run identical training configuration twice and quantify metric deltas for stability validation.

In [ ]:
cmd_stability = (
    'scripts/run_in_pytorch_env.sh python -m src.train.train '
    '--input-csv data/interim/traverse_city_daytime_clean_v1.csv '
    '--image-dir data/processed/images/lake_michigan_64_png '
    '--output-json artifacts/reports/stability_train_report_demo.json '
    '--checkpoint-prefix models/pytorch/michigancast_stability_demo '
    '--stability-runs 2 '
    '--acceptable-delta 0.03 '
    '--device auto'
)
run_or_print(cmd_stability)

### 2.3 TorchScript Export T25

**Purpose** Export the selected checkpoint into a serving friendly runtime format.

In [ ]:
cmd_export = (
    'scripts/run_in_pytorch_env.sh python -m src.train.export '
    '--checkpoint-path models/pytorch/michigancast_multimodal_best.pth '
    '--output-path artifacts/models/michigancast_multimodal.ts '
    '--metadata-path artifacts/models/michigancast_multimodal_metadata.json '
    '--device cpu'
)
run_or_print(cmd_export)

### 2.4 Standalone Inference Validation T25

**Purpose** Confirm that the exported artifact can be loaded outside the training pipeline.

In [ ]:
cmd_infer = (
    'scripts/run_in_pytorch_env.sh python -m src.serve.infer_torchscript '
    '--model-path artifacts/models/michigancast_multimodal.ts '
    '--output-json artifacts/models/inference_output_demo.json '
    '--device cpu'
)
run_or_print(cmd_infer)

### 2.5 Service Startup and Monitoring T35 and T36

Run the API service in a dedicated terminal to avoid notebook blocking:

```bash
scripts/run_in_pytorch_env.sh python -m src.serve.app   --model-path artifacts/models/michigancast_multimodal.ts   --host 127.0.0.1 --port 8011 --device cpu
```

Key endpoints:
- `GET /health`
- `POST /predict`
- `GET /metrics/summary`

## 3. Results and Interpretation

Validate the closure loop with four checks:
1. Imbalance strategy comparison table.
2. Stability report acceptance status.
3. Export metadata consistency.
4. Standalone inference outputs and monitoring summary.

In [ ]:
imb_csv = Path('artifacts/reports/imbalance_strategy_comparison.csv')
imb_json = Path('artifacts/reports/imbalance_strategy_comparison.json')

if imb_csv.exists():
    df_imb = pd.read_csv(imb_csv)
    display(df_imb.sort_values('recall_delta_vs_baseline', ascending=False))
else:
    print('imbalance comparison csv not found. Run section 2.1 first.')

if imb_json.exists():
    payload = json.loads(imb_json.read_text(encoding='utf-8'))
    print('significant improvement:', payload.get('has_significant_positive_metric_improvement'))

In [ ]:
stability_json_candidates = [
    Path('artifacts/reports/stability_train_report_demo.json'),
    Path('artifacts/reports/stability_train_report.json'),
    Path('artifacts/reports/stability_train_report_smoke.json'),
]

found = None
for p in stability_json_candidates:
    if p.exists():
        found = p
        break

if found is not None:
    report = json.loads(found.read_text(encoding='utf-8'))
    print('stability report:', found)
    print('is_stable:', report.get('is_stable'))
    print('max_delta:', report.get('max_delta'))
    print('acceptable_delta:', report.get('acceptable_delta'))
else:
    print('stability report not found. Run section 2.2 first.')

In [ ]:
export_meta = Path('artifacts/models/michigancast_multimodal_metadata.json')
infer_out = Path('artifacts/models/inference_output_demo.json')

if export_meta.exists():
    meta = json.loads(export_meta.read_text(encoding='utf-8'))
    print('torchscript path:', meta.get('torchscript_path'))
    print('trace max abs diff:', meta.get('trace_max_abs_diff'))
else:
    print('export metadata not found. Run section 2.3 first.')

if infer_out.exists():
    infer_payload = json.loads(infer_out.read_text(encoding='utf-8'))
    print('inference probability:', infer_payload.get('rain_probability'))
else:
    print('inference output not found. Run section 2.4 first.')

### 3.1 Presentation Highlights

1. Discuss recall gains versus precision cost under imbalance handling.
2. Explain why stability thresholds are required for production readiness.
3. Show how export, inference, and monitoring complete the delivery loop.

## 4. Engineering Notes and Next Steps

- Add strategy selection policy based on operating precision targets.
- Validate schema compatibility between offline export and online input.
- Add service load testing with latency and distribution drift tracking.

## Appendix. Reproducible Commands

```bash
scripts/run_in_pytorch_env.sh python -m src.train.imbalance --output-dir artifacts/reports
scripts/run_in_pytorch_env.sh python -m src.train.train --stability-runs 2 --output-json artifacts/reports/stability_train_report_demo.json
scripts/run_in_pytorch_env.sh python -m src.train.export --checkpoint-path models/pytorch/michigancast_multimodal_best.pth --output-path artifacts/models/michigancast_multimodal.ts
scripts/run_in_pytorch_env.sh python -m src.serve.infer_torchscript --model-path artifacts/models/michigancast_multimodal.ts --output-json artifacts/models/inference_output_demo.json
scripts/run_in_pytorch_env.sh python -m src.serve.app --model-path artifacts/models/michigancast_multimodal.ts --host 127.0.0.1 --port 8011
```